In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "m5")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def add_ingest_metadata(df, source_file):
    return (df
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(source_file)))

def write_table(df, name):
    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(name))

In [0]:
sales_path = f"{RAW}/sales_train_evaluation.csv"
header = spark.read.option("header", True).csv(sales_path).columns

sales_schema = StructType([
    StructField(c, IntegerType() if c.startswith("d_") else StringType())
    for c in header
])

sales = spark.read.option("header", True).schema(sales_schema).csv(sales_path)
write_table(add_ingest_metadata(sales, "sales_train_evaluation.csv"), "bronze_sales")

In [0]:
for file, table in [("calendar.csv", "bronze_calendar"),
                    ("sell_prices.csv", "bronze_prices")]:
    df = (spark.read.option("header", True)
                    .option("inferSchema", True)
                    .csv(f"{RAW}/{file}"))
    write_table(add_ingest_metadata(df, file), table)

In [0]:
%sql
SELECT 'sales' AS t, COUNT(*) FROM bronze_sales
UNION ALL SELECT 'calendar', COUNT(*) FROM bronze_calendar
UNION ALL SELECT 'prices', COUNT(*) FROM bronze_prices;

t,COUNT(*)
sales,30490
calendar,1969
prices,6841121
